### 09 SQLite에 대화 내용 저장하기

- 사용자가 나갔다가 돌아와서 이전 내용에서 이어나가면서 대화를 하고 싶다면 대화 내용을 데이터베이스에 기록해야한다
- SQLite 데이터베이스를 사용해서 대화 기록을 저장

In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

**사용방법**

- storage를 사용하려면 다음 2가지만 제공하면 된다

1. `session_id` - 사용자 이름, 이메일, 채팅 ID 등과 같은 세션의 고유 식별자

2. `connection` - 데이터베이스 연결을 지정하는 문자열. 이 문자열은 SQLAlchemy의 create_engine 함수에 전달

In [2]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

# SQLChatMessageHistory 객체를 생성하고 세션 ID와 데이터베이스 연결 파일을 설정
chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db"
)

- 여기서 세션이란 여러 질문과 응답이 연속된 하나의 대화 흐름
- 이 코드를 실행하면 대화 내용이 자동으로 데이터베이스에 저장

In [3]:
# 사용자 메시지를 추가합니다.
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!"
)
# AI 메시지를 추가합니다.
chat_message_history.add_ai_message("안녕 테디, 만나서 반가워. 나도 잘 부탁해!")

In [4]:
chat_message_history.messages

[HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={})]

- 사용자별로 세션을 다르게 지정하고 싶으면 `session_id`를 사용자별로 다르게 지정하면 된다
- 이제 Chain에 연결해보자. MessagesPlaceHolder에는 SQL 데이터베이스에서 가져온 대화 내용이 들어간다

In [6]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        # 시스템 메시지
        ("system", "You are a helpful assistant."),
        # 대화 기록을 위한 Placeholder
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),  # 질문
    ]
)

# chain 을 생성합니다.
chain = prompt | ChatOpenAI(model_name="gpt-4.1-mini") | StrOutputParser()

- `sqlite.db` 에서 대화내용을 가져오는 함수

In [7]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name=user_id,
        session_id=conversation_id,
        connection="sqlite:///sqlite.db",
    )

- `config_fields` 를 설정. 이는 대화정보를 조회할 때 참고 정보로 활용
  - `user_id`: 사용자 ID
  - `conversation_id`: 대화 ID

In [8]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id="user_id",
        annotation=str,
        name="User ID",
        description="Unique identifier for a user.",
        default="",
        is_shared=True,
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation.",
        default="",
        is_shared=True,
    ),
]

- `RunnableWithMessageHistory`는 체인과 대화기록을 연동한 후 대화 기록을 참조하여 입력 메시지에 대한 응답을 생성

In [9]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,  # 대화 기록을 가져오는 함수를 설정
    input_messages_key="question",  # 입력 메시지의 키를 "question"으로 설정
    history_messages_key="chat_history",  # 대화 기록 메시지의 키를 "history"로 설정
    history_factory_config=config_fields,  # 대화 기록 조회시 참고할 파라미터를 설정
)

- `"configurable"` 키 아래에 `"user_id"`, `"conversation_id"` key-value 쌍을 설정

In [10]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation1"}}

In [11]:
# 질문과 config 를 전달하여 실행
chain_with_history.invoke({"question": "안녕 반가워, 내 이름은 테디야"}, config)

'안녕 테디야! 반가워. 어떻게 도와줄까?'

In [12]:
# 후속 질문을 실행
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'네 이름은 테디야! 무슨 도움이 필요해?'

- 이번에는 같은 `user_id` 를 가지지만 `conversion_id` 가 다른 값을 가지도록 설정

In [13]:
# config 설정
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation2"}}

# 질문과 config 를 전달하여 실행합니다.
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'죄송하지만, 고객님의 이름을 알 수 없습니다. 알려주시면 기억하겠습니다!'